# Debug Notebook: Compact Ablation (Quick Run)

This notebook isolates dominant factors for speculative decoding performance by varying:
- `k` (proposal length)
- output length (`max_new_tokens`)
- prompt length bucket (short / medium / long)

Design goal: fast, GPU-only debugging run with compact but informative outputs.

In [1]:
import gc
import os
import sys
from pathlib import Path

import pandas as pd
try:
    import torch
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "torch is required for this notebook. Select a kernel/environment with PyTorch installed."
    ) from exc

# Resolve project root robustly.
search_roots = []
if '__file__' in globals():
    search_roots.append(Path(__file__).resolve().parent)
search_roots.append(Path.cwd().resolve())

project_root = None
seen = set()
for start in search_roots:
    for p in [start, *start.parents]:
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src').exists():
            project_root = p
            break
    if project_root is not None:
        break

if project_root is None:
    raise RuntimeError('Could not find project root containing src/.')

src_dir = project_root / 'src'
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Offline-first behavior.
os.environ['SPECDEC_HF_OFFLINE_FIRST'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# GPU-only enforcement.
if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
    raise RuntimeError('CUDA GPU is required for this notebook.')

from config import TARGET_MODEL_ID, DRAFT_MODELS, REGIMES
from baseline import run_baseline_sample
from speculative import load_model_on_device, speculative_decode_sample

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Project root: {project_root}')
print('GPU-only mode enabled')
print(f'CUDA device count: {torch.cuda.device_count()}')

Project root: C:\Working\speculative-decoding-main_v9\speculative-decoding-main
GPU-only mode enabled
CUDA device count: 1


In [2]:
# Ablation controls (compact quick-run).
DEVICE = 'cuda:0'
TARGET_QUANT = 'fp16'
DRAFT_QUANT = 'fp16'
DRAFT_LABEL = '0.5B'
MODES = ['Deterministic']  # add 'Stochastic' if needed

K_VALUES = [2, 4, 8]
OUTPUT_LENGTHS = [12, 32, 64]
PROMPT_BUCKETS = ['short', 'medium', 'long']

BASE_PROMPTS = [
    'Why are leaves green? Answer in one sentence.',
    'What is an API? One-sentence answer.',
    'Define overfitting in machine learning in one sentence.',
    'Give one concise writing tip.',
]

N_PROMPTS_PER_BUCKET = 2

RAW_CSV = results_dir / 'ablation_compact_raw.csv'
SUMMARY_CSV = results_dir / 'ablation_compact_summary.csv'
DOMINANCE_CSV = results_dir / 'ablation_factor_dominance.csv'

print('Ablation settings:')
print(f'  device={DEVICE}, target_quant={TARGET_QUANT}, draft_quant={DRAFT_QUANT}, draft={DRAFT_LABEL}')
print(f'  modes={MODES}, k={K_VALUES}, output_lengths={OUTPUT_LENGTHS}, buckets={PROMPT_BUCKETS}')

Ablation settings:
  device=cuda:0, target_quant=fp16, draft_quant=fp16, draft=0.5B
  modes=['Deterministic'], k=[2, 4, 8], output_lengths=[12, 32, 64], buckets=['short', 'medium', 'long']


In [3]:
# Prompt-bucket constructor (vary prompt length while preserving task intent).
MEDIUM_PREFIX = (
    'Context: In practical ML systems, response quality, latency, and resource usage must be balanced. '
    'Provide a concise, technically accurate answer. '
)
LONG_PREFIX = ' '.join([
    'Context: System constraints include memory bandwidth, kernel launch overhead, and cache behavior.'
    'Prompt engineering can alter prefill cost and token distribution.'
    'Focus on clarity and avoid unnecessary verbosity.'
] * 8)

def make_prompt(base: str, bucket: str) -> str:
    if bucket == 'short':
        return base
    if bucket == 'medium':
        return f'{MEDIUM_PREFIX}\nQuestion: {base}'
    if bucket == 'long':
        return f'{LONG_PREFIX}\nQuestion: {base}'
    raise ValueError(f'Unknown bucket: {bucket}')

bucket_prompts = {}
for b in PROMPT_BUCKETS:
    bucket_prompts[b] = [make_prompt(p, b) for p in BASE_PROMPTS[:N_PROMPTS_PER_BUCKET]]

for b, ps in bucket_prompts.items():
    print(f'{b}: {len(ps)} prompts')

short: 2 prompts
medium: 2 prompts
long: 2 prompts


In [4]:
# Model caches and single-run helper.
target_cache = {}
draft_cache = {}

def _get_target(quant: str):
    key = (TARGET_MODEL_ID, quant.lower(), DEVICE)
    if key not in target_cache:
        target_cache[key] = load_model_on_device(TARGET_MODEL_ID, device=DEVICE, quant_mode=quant.lower())
    return target_cache[key]

def _get_draft(quant: str):
    model_id = DRAFT_MODELS[DRAFT_LABEL]
    key = (model_id, quant.lower(), DEVICE)
    if key not in draft_cache:
        draft_cache[key] = load_model_on_device(model_id, device=DEVICE, quant_mode=quant.lower())
    return draft_cache[key]

def run_one(prompt: str, mode: str, k: int, max_new_tokens: int) -> dict:
    regime = REGIMES[mode.lower()]

    target_model, target_tok = _get_target(TARGET_QUANT)
    draft_model, _ = _get_draft(DRAFT_QUANT)

    # Prompt token length (prefill proxy).
    prompt_tokens = int(target_tok(prompt, return_tensors='pt', truncation=True, max_length=2048)['input_ids'].shape[1])

    base = run_baseline_sample(
        target_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=regime.temperature,
        top_p=regime.top_p,
    )

    spec = speculative_decode_sample(
        target_model,
        draft_model,
        target_tok,
        prompt,
        max_new_tokens=max_new_tokens,
        k=k,
        temperature=regime.temperature,
        top_p=regime.top_p,
        return_timing_breakdown=True,
    )

    base_s = float(base.get('latency_s', 0.0))
    spec_s = float(spec.get('latency_s', 0.0))
    draft_s = float(spec.get('draft_elapsed_s', 0.0))
    verify_s = float(spec.get('verify_elapsed_s', 0.0))
    other_s = max(spec_s - draft_s - verify_s, 0.0)

    proposed = float(spec.get('total_proposed', 0.0))
    accepted = float(spec.get('total_accepted', 0.0))
    alpha = (accepted / proposed) if proposed > 0 else 0.0
    speedup = (base_s / spec_s) if spec_s > 0 else 0.0

    return {
        'prompt_tokens': prompt_tokens,
        'baseline_latency_s': base_s,
        'spec_latency_s': spec_s,
        'speedup': speedup,
        'alpha': alpha,
        'draft_s': draft_s,
        'verify_s': verify_s,
        'other_s': other_s,
        'draft_share': (draft_s / spec_s) if spec_s > 0 else 0.0,
        'verify_share': (verify_s / spec_s) if spec_s > 0 else 0.0,
        'other_share': (other_s / spec_s) if spec_s > 0 else 0.0,
    }

In [5]:
# Execute compact ablation grid.
rows = []
total_runs = len(MODES) * len(K_VALUES) * len(OUTPUT_LENGTHS) * len(PROMPT_BUCKETS) * N_PROMPTS_PER_BUCKET
done = 0

for mode in MODES:
    for k in K_VALUES:
        for out_len in OUTPUT_LENGTHS:
            for bucket in PROMPT_BUCKETS:
                prompts = bucket_prompts[bucket]
                for i, prompt in enumerate(prompts, start=1):
                    done += 1
                    print(f'[{done}/{total_runs}] mode={mode} k={k} out={out_len} bucket={bucket} sample={i}')
                    m = run_one(prompt=prompt, mode=mode, k=k, max_new_tokens=out_len)
                    rows.append({
                        'mode': mode,
                        'k': k,
                        'max_new_tokens': out_len,
                        'prompt_bucket': bucket,
                        'sample_idx': i,
                        **m,
                    })

df_raw = pd.DataFrame(rows)
df_raw.to_csv(RAW_CSV, index=False)
print(f'Saved raw runs -> {RAW_CSV}')
display(df_raw.head())

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

[1/54] mode=Deterministic k=2 out=12 bucket=short sample=1
Loading model: Qwen/Qwen2.5-3B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading model: Qwen/Qwen2.5-0.5B-Instruct (device=cuda:0, quant=fp16 -> fp16, offline_first=True)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[2/54] mode=Deterministic k=2 out=12 bucket=short sample=2
[3/54] mode=Deterministic k=2 out=12 bucket=medium sample=1
[4/54] mode=Deterministic k=2 out=12 bucket=medium sample=2
[5/54] mode=Deterministic k=2 out=12 bucket=long sample=1
[6/54] mode=Deterministic k=2 out=12 bucket=long sample=2
[7/54] mode=Deterministic k=2 out=32 bucket=short sample=1
[8/54] mode=Deterministic k=2 out=32 bucket=short sample=2
[9/54] mode=Deterministic k=2 out=32 bucket=medium sample=1
[10/54] mode=Deterministic k=2 out=32 bucket=medium sample=2
[11/54] mode=Deterministic k=2 out=32 bucket=long sample=1
[12/54] mode=Deterministic k=2 out=32 bucket=long sample=2
[13/54] mode=Deterministic k=2 out=64 bucket=short sample=1
[14/54] mode=Deterministic k=2 out=64 bucket=short sample=2
[15/54] mode=Deterministic k=2 out=64 bucket=medium sample=1
[16/54] mode=Deterministic k=2 out=64 bucket=medium sample=2
[17/54] mode=Deterministic k=2 out=64 bucket=long sample=1
[18/54] mode=Deterministic k=2 out=64 bucket=lo

,mode,k,max_new_tokens,prompt_bucket,sample_idx,prompt_tokens,baseline_latency_s,spec_latency_s,speedup,alpha,draft_s,verify_s,other_s,draft_share,verify_share,other_share
0,Deterministic,2,12,short,1,10,0.7026,0.4435,1.584216,0.384615,0.238829,0.202878,0.001793,0.538510,0.457448,0.004043
1,Deterministic,2,12,short,2,10,0.3661,0.3765,0.972377,0.545455,0.206274,0.168839,0.001387,0.547873,0.448444,0.003684
2,Deterministic,2,12,medium,1,40,0.3567,0.5234,0.681506,0.266667,0.296170,0.225143,0.002087,0.565858,0.430155,0.003987
3,Deterministic,2,12,medium,2,40,0.3548,0.3097,1.145625,0.777778,0.169391,0.139328,0.000981,0.546952,0.449881,0.003168
4,Deterministic,2,12,long,1,284,0.3766,0.4895,0.769356,0.357143,0.274006,0.213674,0.001820,0.559767,0.436515,0.003718


In [6]:
# Compact ablation summary table.
group_cols = ['mode', 'k', 'max_new_tokens', 'prompt_bucket']
summary = (
    df_raw.groupby(group_cols, as_index=False)
    .agg(
        n=('sample_idx', 'count'),
        prompt_tokens_mean=('prompt_tokens', 'mean'),
        speedup_mean=('speedup', 'mean'),
        alpha_mean=('alpha', 'mean'),
        spec_latency_mean_s=('spec_latency_s', 'mean'),
        baseline_latency_mean_s=('baseline_latency_s', 'mean'),
        draft_share_mean=('draft_share', 'mean'),
        verify_share_mean=('verify_share', 'mean'),
        other_share_mean=('other_share', 'mean'),
    )
)

for c in ['draft_share_mean', 'verify_share_mean', 'other_share_mean']:
    summary[c] = summary[c] * 100.0

summary.to_csv(SUMMARY_CSV, index=False)
print(f'Saved summary -> {SUMMARY_CSV}')
display(summary.sort_values(['mode', 'k', 'max_new_tokens', 'prompt_bucket']))

Saved summary -> C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\ablation_compact_summary.csv


,mode,k,max_new_tokens,prompt_bucket,n,prompt_tokens_mean,speedup_mean,alpha_mean,spec_latency_mean_s,baseline_latency_mean_s,draft_share_mean,verify_share_mean,other_share_mean
0,Deterministic,2,12,long,2,284.0,0.929683,0.528571,0.41640,0.37540,55.675073,43.988718,0.336210
1,Deterministic,2,12,medium,2,40.0,0.913565,0.522222,0.41655,0.35575,55.640487,44.001764,0.357749
2,Deterministic,2,12,short,2,10.0,1.278297,0.465035,0.41000,0.53435,54.319105,45.294557,0.386339
3,Deterministic,2,32,long,2,284.0,0.800343,0.365497,1.13665,0.90940,56.955118,42.680650,0.364232
4,Deterministic,2,32,medium,2,40.0,0.759632,0.329522,1.20395,0.91420,57.259880,42.341989,0.398132
5,Deterministic,2,32,short,2,10.0,0.781079,0.371053,1.17435,0.91375,57.266294,42.353577,0.380129
6,Deterministic,2,64,long,2,284.0,0.832564,0.437812,2.14560,1.77465,56.877506,42.784811,0.337683
7,Deterministic,2,64,medium,2,40.0,0.791834,0.334331,2.54090,2.00315,57.973694,41.650644,0.375662
8,Deterministic,2,64,short,2,10.0,0.765344,0.388889,2.46250,1.87375,58.177076,41.463787,0.359137
9,Deterministic,4,12,long,2,284.0,0.739952,0.416409,0.47520,0.35120,67.158529,32.588301,0.253170


In [7]:
# Dominance scores: how much variance each factor explains (main effects only).
def dominance_score(df: pd.DataFrame, y: str, factor: str) -> float:
    total_var = float(df[y].var(ddof=0))
    if total_var <= 0:
        return 0.0
    between = float(df.groupby(factor)[y].mean().var(ddof=0))
    return between / total_var

factors = ['k', 'max_new_tokens', 'prompt_bucket']
targets = ['speedup', 'spec_latency_s', 'alpha']
dom_rows = []
for y in targets:
    for f in factors:
        dom_rows.append({
            'metric': y,
            'factor': f,
            'dominance_score': dominance_score(df_raw, y, f),
        })

df_dom = pd.DataFrame(dom_rows)
df_dom.to_csv(DOMINANCE_CSV, index=False)
print(f'Saved dominance -> {DOMINANCE_CSV}')
display(df_dom.sort_values(['metric', 'dominance_score'], ascending=[True, False]))

print('\nHigher dominance_score means stronger main-effect contribution to variance.')

Saved dominance -> C:\Working\speculative-decoding-main_v9\speculative-decoding-main\results\ablation_factor_dominance.csv


,metric,factor,dominance_score
6,alpha,k,0.216530
7,alpha,max_new_tokens,0.211781
8,alpha,prompt_bucket,0.009421
4,spec_latency_s,max_new_tokens,0.797665
3,spec_latency_s,k,0.095017
5,spec_latency_s,prompt_bucket,0.009364
0,speedup,k,0.414328
1,speedup,max_new_tokens,0.181499
2,speedup,prompt_bucket,0.009835



Higher dominance_score means stronger main-effect contribution to variance.


## Notes

- This notebook is for compact diagnosis, not final benchmark claims.
- To run faster: reduce `N_PROMPTS_PER_BUCKET` or fewer `OUTPUT_LENGTHS`.
- To compare regimes, set `MODES = ['Deterministic', 'Stochastic']`.
- You can reuse outputs in slides directly from:
  - `results/ablation_compact_summary.csv`
  - `results/ablation_factor_dominance.csv`